# 🚢 Session 7 Lab: Interactive Titanic (Solutions)

**Prerequisite:** `01_Plotly_Tutorial.ipynb`

Solved exercises and model answers. The 💡 ideas are solved here too.

| Part | Exercises |
|---|---|
| A: One column at a time | 1, 2, 3 |
| B: Comparing groups | 4, 5, 6 |
| C: Relationships | 7, 8 |
| D: The big picture and sharing | 9, 10, 11 |

In [2]:
from pathlib import Path
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "plotly_mimetype+notebook_connected"   # plotly.js is loaded from the internet: the saved notebook stays small

df = pd.read_csv("data/titanic - titanic (1).csv")
df.columns = df.columns.str.lower()
df["class"] = df["pclass"].map({1: "First", 2: "Second", 3: "Third"})
df["survival"] = df["survived"].map({0: "Died", 1: "Survived"})
df["port"] = df["embarked"].map({"C": "Cherbourg", "Q": "Queenstown", "S": "Southampton"})
df["family_size"] = df["sibsp"] + df["parch"] + 1
df["title"] = df["name"].str.extract(r",\s*([^\.]+)\.", expand=False)
df["age_group"] = pd.cut(df["age"], [0, 12, 18, 35, 60, 80],
                         labels=["child", "teen", "young adult", "adult", "senior"])

ORDERS = {"class": ["First", "Second", "Third"], "survival": ["Died", "Survived"],
          "age_group": ["child", "teen", "young adult", "adult", "senior"]}
COLORS = {"Died": "#D95D4A", "Survived": "#0F766E"}
df.head(3)

,passengerid,survived,pclass,name,sex,age,sibsp,parch,ticket,fare,cabin,embarked,class,survival,port,family_size,title,age_group
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S,Third,Died,Southampton,2,Mr,young adult
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C,First,Survived,Cherbourg,2,Mrs,adult
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S,Third,Survived,Southampton,1,Miss,young adult


The setup made the readable columns for you: `class`, `survival`, `port`, `family_size`, `title`, `age_group`, plus a fixed order (`ORDERS`) and colours (`COLORS`).

---
# A: One Column at a Time

### 1. Family size
Draw a histogram of `family_size`, one bar per size. Hover the bars: which size is the most common, and how many passengers have it?

In [2]:
fig = px.histogram(df, x="family_size", nbins=11, labels={"family_size": "Family size (incl. the passenger)"},
                   title="How big were the families on board?")
fig.update_layout(bargap=0.1, yaxis_title="Passengers")
fig.show()

❓ **Question:** Which family size is the most common, and how many passengers?

✍️ **Answer:** Travelling alone (size 1): 537 of the 891 passengers, 60%. Sizes above 4 are rare.

### 2. Age
A histogram of `age` with a box plot on top of it.

In [3]:
fig = px.histogram(df, x="age", nbins=30, marginal="box",
                   labels={"age": "Age"}, title="Age of the passengers")
fig.show()

> 💡 **Try it another way:** draw the same with `marginal="violin"`. What does the violin add to the box?

In [4]:
fig = px.histogram(df, x="age", nbins=30, marginal="violin",
                   labels={"age": "Age"}, title="Age of the passengers")
fig.show()

The violin shows the same shape as the histogram, but smoothed, so you see where the passengers bunch up (the 20s and 30s). The box only gives the quartiles.

### 3. Fare by class
A box plot of `fare` for each `class`, each class in its own colour, classes in order. Draw it once as it is, and once on a **log** y-axis.

In [5]:
fig = px.box(df, x="class", y="fare", color="class", category_orders=ORDERS,
             labels={"class": "Class", "fare": "Fare"}, title="Fare by class")
fig.show()

fig = px.box(df, x="class", y="fare", color="class", category_orders=ORDERS, log_y=True,
             labels={"class": "Class", "fare": "Fare (log scale)"}, title="Fare by class (log scale)")
fig.show()

❓ **Question:** Hover the boxes. What is the median fare in each class? Why is the log version easier to read?

✍️ **Answer:** First 60.3, Second 14.3, Third 8.05. On the linear axis a few First-class fares above 500 squeeze the Second and Third boxes into flat lines at the bottom. The log axis spreads the small fares out, so all three boxes are readable.

---
# B: Comparing Groups

### 4. The most common titles
`top_titles`: the 6 most common values of `title` with their counts. Then a bar chart of them, sorted, with the count written on each bar.

In [6]:
top_titles = df["title"].value_counts().head(6).reset_index()

fig = px.bar(top_titles, x="title", y="count", text_auto=True,
             labels={"title": "Title", "count": "Passengers"}, title="The 6 most common titles")
fig.show()

### 5. Overall survival
A donut chart of how many passengers died and survived, in the colours of `COLORS`.

In [7]:
outcome = df["survival"].value_counts().reset_index()

fig = px.pie(outcome, names="survival", values="count", hole=0.5, color="survival",
             color_discrete_map=COLORS, title="Died vs survived")
fig.show()

### 6. Port and survival
`port_counts`: the number of passengers for each `port` and `survival`. Then a **grouped** bar chart of it, coloured by survival.

In [8]:
port_counts = df.groupby(["port", "survival"]).size().reset_index(name="passengers")

fig = px.bar(port_counts, x="port", y="passengers", color="survival", barmode="group",
             color_discrete_map=COLORS, category_orders=ORDERS,
             labels={"port": "Port", "passengers": "Passengers", "survival": ""},
             title="Passengers by port and outcome")
fig.show()

❓ **Question:** At which port did more passengers survive than die?

✍️ **Answer:** Only at Cherbourg: 93 survived, 75 died. At Southampton almost twice as many died as survived (427 vs 217).

---
# C: Relationships

### 7. Age, fare and class
A scatter of `age` (x) against `fare` (y):
- coloured by `survival`, one panel per `class` (in order), fare on a log axis
- the hover shows the passenger's **name** in bold, plus `sex` and `family_size`
- the panel titles say only *First*, *Second*, *Third*

In [9]:
fig = px.scatter(df, x="age", y="fare", color="survival", facet_col="class", log_y=True, opacity=0.7,
                 color_discrete_map=COLORS, category_orders=ORDERS,
                 hover_name="name", hover_data=["sex", "family_size"],
                 labels={"age": "Age", "fare": "Fare (log)", "survival": ""},
                 title="Age vs fare, by class")
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.show()

❓ **Question:** In which panel are the survivors the majority? Hover one of the oldest survivors: who is it?

✍️ **Answer:** In the First-class panel the green dots are the majority (63% survived); in Third class the red dots dominate. The oldest survivor is Mr. Algernon Henry Wilson Barkworth, 80, First class.

### 8. Survival rate heatmap
`rate`: a table of the survival rate, `sex` as rows and `class` as columns. Show it as a heatmap with the rates written in the cells as percentages, and the colour scale from 0 to 1.

In [10]:
rate = df.pivot_table(index="sex", columns="class", values="survived", aggfunc="mean")

fig = px.imshow(rate, text_auto=".0%", zmin=0, zmax=1, color_continuous_scale="Teal",
                labels={"x": "Class", "y": "Sex", "color": "Survival rate"},
                title="Survival rate by sex and class")
fig.show()

> 💡 **Try it another way:** the same heatmap with `age_group` as rows. And a correlation heatmap of the numeric columns, colour scale from −1 to 1.

In [11]:
rate_age = df.pivot_table(index="age_group", columns="class", values="survived", aggfunc="mean", observed=True)
fig = px.imshow(rate_age, text_auto=".0%", zmin=0, zmax=1, color_continuous_scale="Teal",
                labels={"x": "Class", "y": "Age group", "color": "Survival rate"},
                title="Survival rate by age group and class")
fig.show()

corr = df[["survived", "pclass", "age", "sibsp", "parch", "fare", "family_size"]].corr()
fig = px.imshow(corr, text_auto=".2f", zmin=-1, zmax=1, color_continuous_scale="RdBu_r",
                title="Correlations")
fig.show()

All 17 children in Second class survived, and 3 of the 4 in First, but only 42% of the children in Third class. In the correlation heatmap, `survived` goes most strongly with `pclass` (−0.34) and `fare` (+0.26); sex isn't there because it isn't a number yet.

❓ **Question:** Which group had the highest survival rate, and which the lowest?

✍️ **Answer:** Women in First class: 97%. Men in Third class: 14%. Even women in Third class (50%) did better than men in First class (37%).

---
# D: The Big Picture and Sharing

### 9. Passenger treemap
`groups`: one row per `class` and `sex` with the number of passengers (`passengers`) and the survival rate (`rate`). Then a treemap class → sex: the box size is the number of passengers, the colour is the survival rate.

In [12]:
groups = (df.groupby(["class", "sex"])
            .agg(passengers=("survived", "size"), rate=("survived", "mean")).reset_index())

fig = px.treemap(groups, path=["class", "sex"], values="passengers", color="rate",
                 color_continuous_scale="Teal", range_color=[0, 1],
                 title="Passengers (size) and survival rate (colour)")
fig.show()

> 💡 **Try it another way:** the same `groups` as a `px.sunburst`.

In [13]:
fig = px.sunburst(groups, path=["class", "sex"], values="passengers", color="rate",
                  color_continuous_scale="Teal", range_color=[0, 1], title="The same, as a sunburst")
fig.show()

### 10. Titles as bubbles
For the titles `Mr`, `Mrs`, `Miss`, `Master`: `titles` holds the mean age, the survival rate and the number of passengers of each. Then a bubble chart: x = mean age, y = survival rate, size = passengers, colour = title.

In [14]:
titles = (df[df["title"].isin(["Mr", "Mrs", "Miss", "Master"])]
            .groupby("title")
            .agg(mean_age=("age", "mean"), rate=("survived", "mean"), passengers=("survived", "size"))
            .reset_index())

fig = px.scatter(titles, x="mean_age", y="rate", size="passengers", color="title", size_max=60,
                 hover_name="title", labels={"mean_age": "Mean age", "rate": "Survival rate"},
                 title="Titles: age, survival and size")
fig.show()
titles

,title,mean_age,rate,passengers
0,Master,4.574167,0.575000,40
1,Miss,21.773973,0.697802,182
2,Mr,32.368090,0.156673,517
3,Mrs,35.898148,0.792000,125


❓ **Question:** *Master* were boys. How does their survival rate compare with *Mr*?

✍️ **Answer:** Master (boys, mean age about 5) survived 57.5% of the time, Mr only 15.7%. Being a child helped men a lot: "women and children first".

### 11. Share it
Save the treemap from exercise 9 as an interactive web page `figures/titanic_treemap.html`. Then list the files in `figures/` with their size.

In [15]:
Path("figures").mkdir(exist_ok=True)
fig = px.treemap(groups, path=["class", "sex"], values="passengers", color="rate",
                 color_continuous_scale="Teal", range_color=[0, 1],
                 title="Passengers (size) and survival rate (colour)")
fig.write_html("figures/titanic_treemap.html")

for f in Path("figures").glob("*.html"):
    print(f.name, round(f.stat().st_size / 1e6, 1), "MB")

titanic_treemap.html 4.6 MB


❓ **Question:** When would you send a Plotly HTML file instead of a Seaborn PNG?

✍️ **Answer:** When the reader will explore: hover for exact numbers, zoom, hide groups, click into a treemap. For a printed report or a slide, a PNG is simpler and always looks the same.

---
# 🔭 Try After the Session
Add the 💡 ideas you skipped. Then take chart 7 and add `animation_frame="class"` instead of `facet_col`: is a slider or three side-by-side panels easier to compare?

In [16]:
fig = px.scatter(df, x="age", y="fare", color="survival", animation_frame="class", log_y=True,
                 color_discrete_map=COLORS, category_orders=ORDERS, range_x=[0, 82], range_y=[1, 600],
                 hover_name="name", title="Age vs fare, one class per frame")
fig.show()

Side by side is easier: you see the three classes at once. An animation makes the reader remember the previous frame. Keep `animation_frame` for something that really moves in order, such as time.